# Aurora · Colab GPU worker

Turn a Colab GPU into an Aurora backend (lipsync by default — fits the free tier).

**Before running:**
1. **Runtime → Change runtime type** → Hardware accelerator = **GPU**.
2. **Secrets** (key icon, left sidebar): add `NGROK_AUTHTOKEN`, `NGROK_STATIC_DOMAIN`, `AURORA_URL`, `AURORA_REGISTER_SECRET` (optional: `AURORA_WORKER_TOKEN`, `AURORA_TASKS`, `AURORA_WORKER_REPO_RAW`) and toggle **notebook access ON** for each.
3. **Runtime → Run all**.

See `workers/colab/README.md` for where to get each secret. The worker auto-registers in **Admin → Workers** and serves lip-sync jobs for free.

> `motion` (MimicMotion) needs a ≥24 GB GPU (Colab Pro+ A100) — not the free tier. `lipsync` fits any free Colab GPU.

In [ ]:
# Aurora · Colab GPU worker — one-cell launcher.
# Runtime -> Change runtime type -> GPU. Secrets (key icon): NGROK_AUTHTOKEN,
# NGROK_STATIC_DOMAIN, AURORA_URL, AURORA_REGISTER_SECRET (+ optional
# AURORA_WORKER_TOKEN, AURORA_TASKS). Toggle notebook access ON for each, then
# Runtime -> Run all. This fetches the launcher from your repo and runs it.
import os, urllib.request

# For a renamed repo, a non-default branch, or a public mirror, set this secret to
# your raw base, e.g. https://raw.githubusercontent.com/OWNER/REPO/BRANCH/workers
# (a private repo won't fetch over raw URLs — upload the worker files instead).
try:
    from google.colab import userdata
    try:
        _v = userdata.get("AURORA_WORKER_REPO_RAW")
        if _v:
            os.environ["AURORA_WORKER_REPO_RAW"] = _v.strip()
    except Exception:
        pass
except Exception:
    pass

_explicit = os.environ.get("AURORA_WORKER_REPO_RAW", "").strip().rstrip("/")
_bases = [_explicit] if _explicit else [
    f"https://raw.githubusercontent.com/josephcrown920/Auroraglobal/{b}/workers"
    for b in ("Main", "main", "master")
]
_dst, _err = "/content/aurora_worker_colab.py", None
for _b in _bases:
    try:
        urllib.request.urlretrieve(f"{_b}/colab/aurora_worker_colab.py", _dst)
        break
    except Exception as e:
        _err = e
else:
    raise SystemExit(
        f"Could not fetch launcher from {_bases}: {_err}\n"
        "Set the AURORA_WORKER_REPO_RAW secret or make the repo public."
    )
exec(open(_dst).read())
